In [ ]:
!pip install ultralytics supervision

In [ ]:
# step_1_run_tracking.py (Final "No Guessing" Version)
import os
import glob
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import config
import shutil
import re

def sanitize_filename(name):
    """Creates a safe name for directories."""
    name = re.sub(r'[^\w\.\-]', '_', name)
    return name

def main():
    print("--- Step 1: Running Object Tracking (Final No-Guessing Version) ---")
    Path(config.TRACKING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    
    model = YOLO(config.YOLO_MODEL_PATH)
    
    video_files = glob.glob(os.path.join(config.VIDEO_INPUT_DIR, "*.mp4"))
    
    for video_path_str in tqdm(video_files, desc="Processing Videos"):
        video_path = Path(video_path_str)
        original_clip_name = video_path.stem
        
        safe_clip_name_for_dir = sanitize_filename(original_clip_name)
        temp_run_dir = Path(config.OUTPUT_DIR) / f"{safe_clip_name_for_dir}_tracking_run"
        
        results_generator = model.track(
            source=video_path_str,
            tracker='botsort.yaml',
            save_txt=True,
            stream=True,
            project=config.OUTPUT_DIR,
            name=f"{safe_clip_name_for_dir}_tracking_run",
            exist_ok=True,
            verbose=False
        )
        
        for _ in results_generator:
            pass
        
        tracks_dir = temp_run_dir / "tracks"
        
        # --- THE FINAL, ROBUST FIX: STOP GUESSING, START SEARCHING ---
        # Search for any .txt file within the tracks directory.
        found_files = glob.glob(str(tracks_dir / "*.txt"))
        
        if found_files:
            # Get the path of the first (and only) .txt file we find.
            track_txt_path = found_files[0]
            
            all_tracks = []
            with open(track_txt_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 6: continue
                    frame_idx, track_id, x_min, y_min, w, h = map(float, parts[:6])
                    all_tracks.append({
                        'frame': int(frame_idx) - 1, 'track_id': int(track_id),
                        'x_min': x_min, 'y_min': y_min, 'x_max': x_min + w, 'y_max': y_min + h
                    })

            if all_tracks:
                output_csv_path = Path(config.TRACKING_OUTPUT_DIR) / f"{original_clip_name}_tracks.csv"
                df = pd.DataFrame(all_tracks)
                df.to_csv(output_csv_path, index=False)
            else:
                print(f"Warning: Track file was found but was empty for {original_clip_name}")
        else:
            print(f"FATAL: No .txt track file was created in '{tracks_dir}' for video '{original_clip_name}'.")

        if temp_run_dir.exists():
            shutil.rmtree(temp_run_dir)

    print("\n--- Tracking Complete ---")
    print(f"All tracking CSVs have been saved to: {config.TRACKING_OUTPUT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
with open("/kaggle/working/config.py", 'r') as f:
    print(f.read())

In [ ]:
import shutil
import os

# Define the specific file and folders
source_file = r'/kaggle/input/acc-sample-50-config-file/config.py'
destination_folder = r'/kaggle/working/'

# Create destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Define destination path
destination_path = os.path.join(destination_folder, os.path.basename(source_file))

# Copy the file
shutil.copy2(source_file, destination_path)

print(f"Copied {source_file} to {destination_path}")

In [ ]:

config_file_path = '/kaggle/working/config.py'
new_image_size = (224, 224)
new_clip_length = 32
new_batch_size = 8# A safe starting point for ViT
new_lr = 0.0001
new_patience = 15
new_VIDEO_INPUT_DIR = "/kaggle/input/acc-dataset/total"
new_YOLO_MODEL_PATH = 'yolov9e'

print("--- Modifying config.py for the Vision Transformer (ViT) experiment ---")

if not os.path.exists(config_file_path):
    print(f"FATAL: The file '{config_file_path}' was not found.")
else:
    with open(config_file_path, 'r') as f:
        lines = f.readlines()

    updates = {'IMAGE_SIZE': False, 'CLIP_LENGTH': False, 'BATCH_SIZE': False}

    for i, line in enumerate(lines):
        if line.strip().startswith('IMAGE_SIZE'):
            lines[i] = f'IMAGE_SIZE = {new_image_size}   # H, W for the model\n'
            updates['IMAGE_SIZE'] = True
        elif line.strip().startswith('CLIP_LENGTH'):
            lines[i] = f'CLIP_LENGTH = {new_clip_length}          # Frames per clip\n'
            updates['CLIP_LENGTH'] = True
        elif line.strip().startswith('BATCH_SIZE'):
            lines[i] = f'BATCH_SIZE = {new_batch_size}\n'
            updates['BATCH_SIZE'] = True
        elif line.strip().startswith('LEARNING_RATE'):
            lines[i] = f'LEARNING_RATE = {new_lr}\n'
            updates['LEARNING_RATE'] = True
        elif line.strip().startswith('PATIENCE'):
            lines[i] = f'PATIENCE = {new_patience}\n'
            updates['PATIENCE'] = True
        elif line.strip().startswith('VIDEO_INPUT_DIR'):
            lines[i] = f'VIDEO_INPUT_DIR = "{new_VIDEO_INPUT_DIR}"\n'
            updates['VIDEO_INPUT_DIR'] = True
        elif line.strip().startswith('YOLO_MODEL_PATH'):
            lines[i] = f'YOLO_MODEL_PATH = "{new_YOLO_MODEL_PATH}"\n'
            updates['YOLO_MODEL_PATH'] = True

    if all(updates.values()):
        with open(config_file_path, 'w') as f:
            f.writelines(lines)
        print("✓ Successfully updated 'config.py'.")
    else:
        print("Warning: Could not find all required settings.")

print("\n--- Verifying final content of config.py ---")
with open(config_file_path, 'r') as f:
    print(f.read())

In [ ]:
# step_1_run_tracking.py (Final Programmatic Version)
import os
import glob
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import config
import shutil

def main():
    print("--- Step 1: Running Object Tracking (Programmatic & Immune to Race Conditions) ---")
    Path(config.TRACKING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    
    model = YOLO(config.YOLO_MODEL_PATH)
    
    video_files = glob.glob(os.path.join(config.VIDEO_INPUT_DIR, "*.mp4"))
    
    for video_path_str in tqdm(video_files, desc="Processing Videos"):
        video_path = Path(video_path_str)
        original_clip_name = video_path.stem
        
        # We set stream=True for memory efficiency.
        results_generator = model.track(
            source=video_path_str,
            tracker='botsort.yaml',
            stream=True,
            verbose=False
            # NOTE: We have REMOVED save_txt, project, and name as we are not saving files.
        )
        
        all_tracks_for_video = []
        frame_index = 0
        
        # This loop processes results as they become available.
        for r in results_generator:
            # Check if there are any tracked boxes in the current frame.
            if r.boxes.id is not None:
                # Get the track IDs and bounding boxes (xyxy format).
                track_ids = r.boxes.id.int().cpu().tolist()
                boxes_xyxy = r.boxes.xyxy.cpu().tolist()
                
                # For each detected object, record its data.
                for track_id, box in zip(track_ids, boxes_xyxy):
                    x_min, y_min, x_max, y_max = box
                    all_tracks_for_video.append({
                        'frame': frame_index,
                        'track_id': track_id,
                        'x_min': x_min,
                        'y_min': y_min,
                        'x_max': x_max,
                        'y_max': y_max
                    })
            
            frame_index += 1 # Manually increment the frame counter.

        if all_tracks_for_video:
            df = pd.DataFrame(all_tracks_for_video)
            output_csv_path = Path(config.TRACKING_OUTPUT_DIR) / f"{original_clip_name}_tracks.csv"
            df.to_csv(output_csv_path, index=False)
        else:
            print(f"Warning: No tracks were detected for video {original_clip_name}")

    print("\n--- Tracking Complete ---")
    print(f"All tracking CSVs have been saved to: {config.TRACKING_OUTPUT_DIR}")

if __name__ == "__main__":
    main()

In [ ]:
# step_2_generate_predictions.py
import os
import glob
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import itertools
import config

def calculate_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1_p, y1_p, x2_p, y2_p = box2
    inter_x1, inter_y1 = max(x1, x1_p), max(y1, y1_p)
    inter_x2, inter_y2 = min(x2, x2_p), min(y2, y2_p)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2_p - x1_p) * (y2_p - y1_p)
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def save_frame_from_video(video_path, frame_number, output_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    if ret: cv2.imwrite(str(output_path), frame)
    cap.release()

def main():
    print("--- Step 2: Generating Event Predictions from Tracking Data ---")
    predictions = []
    
    tracking_files = glob.glob(os.path.join(config.TRACKING_OUTPUT_DIR, "*_tracks.csv"))
    verification_dir = Path(config.OUTPUT_DIR) / 'verification_frames'
    verification_dir.mkdir(exist_ok=True, parents=True)

    for track_file in tqdm(tracking_files, desc="Analyzing Tracks"):
        clip_name = Path(track_file).stem.replace('_tracks', '')
        df = pd.read_csv(track_file)
        
        if df.empty: continue
            
        df = df.sort_values(by=['track_id', 'frame']).copy()
        df['x_center'] = (df['x_min'] + df['x_max']) / 2
        df['y_center'] = (df['y_min'] + df['y_max']) / 2
        
        grouped = df.groupby('track_id')
        df[['dx', 'dy']] = grouped[['x_center', 'y_center']].diff().fillna(0)
        df['velocity'] = np.sqrt(df['dx']**2 + df['dy']**2)
        df['deceleration'] = -grouped['velocity'].diff().fillna(0)
        
        frame_features = df.groupby('frame').agg(max_deceleration=('deceleration', 'max')).reset_index()

        iou_data = []
        for frame_num, frame_df in df.groupby('frame'):
            max_iou = 0
            if len(frame_df) > 1:
                for i, j in itertools.combinations(frame_df.index, 2):
                    iou = calculate_iou(frame_df.loc[i, ['x_min', 'y_min', 'x_max', 'y_max']].values, frame_df.loc[j, ['x_min', 'y_min', 'x_max', 'y_max']].values)
                    if iou > max_iou: max_iou = iou
            iou_data.append({'frame': frame_num, 'max_iou': max_iou})
        
        frame_features = pd.merge(frame_features, pd.DataFrame(iou_data), on='frame', how='left').fillna(0)

        max_d = frame_features['max_deceleration'].max()
        max_i = frame_features['max_iou'].max()
        if max_d > 0 or max_i > 0:
            frame_features['anomaly_score'] = 0.7 * (frame_features['max_deceleration'] / max_d if max_d > 0 else 0) + \
                                              0.3 * (frame_features['max_iou'] / max_i if max_i > 0 else 0)
        else:
            frame_features['anomaly_score'] = 0

        if frame_features.empty: continue
        
        peak_frame = frame_features.loc[frame_features['anomaly_score'].idxmax()]['frame']
        start_frame = max(0, int(peak_frame - config.CLIP_LENGTH * 0.5))
        end_frame = int(peak_frame + config.CLIP_LENGTH * 0.5)

        predictions.append({
            'clip_name': clip_name,
            'predicted_start_frame': start_frame,
            'predicted_end_frame': end_frame,
            'peak_frame_for_verification': int(peak_frame)
        })

        video_path = Path(config.VIDEO_INPUT_DIR) / f"{clip_name}.mp4"
        if video_path.exists():
            save_frame_from_video(video_path, int(peak_frame), verification_dir / f"{clip_name}_peak.jpg")

    pred_df = pd.DataFrame(predictions)
    pred_path = Path(config.OUTPUT_DIR) / 'predicted_events.csv'
    pred_df.to_csv(pred_path, index=False)
    
    print(f"\n✓ Predictions saved to {pred_path}")
    print(f"✓ Verification frames saved to {verification_dir}")
    print("--- Prediction Generation Complete ---")

if __name__ == "__main__":
    main()

In [ ]:
import shutil

# Specify the folder to be zipped
folder_to_zip = '/kaggle/working/output/tracking_data'

# Specify the output ZIP file name (without extension)
output_zip = '/kaggle/working/tracking_data'

# Create the ZIP archive
shutil.make_archive(output_zip, 'zip', folder_to_zip)

print(f"Folder '{folder_to_zip}' has been zipped as '{output_zip}.zip'")

In [ ]:
# ===================================================================================
# PREREQUISITE SCRIPT: Video Frame Extractor
# ===================================================================================
import os
import glob
from pathlib import Path
import cv2
from tqdm import tqdm

# --- Configuration ---
VIDEO_INPUT_DIR = "/kaggle/input/acc-sample-50"
FRAMES_OUTPUT_DIR = "/kaggle/working/output_frames/"

def main_extractor():
    print(f"Starting frame extraction from: {VIDEO_INPUT_DIR}")
    Path(FRAMES_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    video_paths = glob.glob(os.path.join(VIDEO_INPUT_DIR, "**/*.mp4"), recursive=True)
    print(f"Found {len(video_paths)} videos to process.")
    for video_path in tqdm(video_paths, desc="Extracting frames"):
        clip_name = Path(video_path).stem
        clip_output_dir = Path(FRAMES_OUTPUT_DIR) / clip_name
        clip_output_dir.mkdir(exist_ok=True)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened(): continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for i in range(frame_count):
            ret, frame = cap.read()
            if not ret: break
            cv2.imwrite(str(clip_output_dir / f"{i:05d}.jpg"), frame)
        cap.release()
    print(f"\n--- Frame Extraction Complete ---")
    print(f"All frames saved to: {FRAMES_OUTPUT_DIR}")

if __name__ == "__main__":
    main_extractor()
    